# Evidências de validação — robô de triagem de pedidos de serviços públicos

Este notebook reúne as evidências pedidas no enunciado (§5): execução com dados de teste, **resultado medido** em
uma amostra rotulada, **análise dos erros**, **casos de falha tratados** e **limites** da solução.

Ele usa os pacotes de `src/triage` (não reimplementa lógica) e roda do início ao fim com
`jupyter nbconvert --to notebook --execute notebooks/evidence.ipynb`. Todos os dados são sintéticos; nenhuma
chave de API é usada e o LLM fica **desligado** em todas as seções.

In [1]:
import dataclasses
import logging
import os
import shutil
import sys
from pathlib import Path
from unittest.mock import MagicMock, patch

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # os caminhos de data/ e do léxico de urgência são relativos à raiz do repositório

import run  # ponto de entrada `python run.py`
from triage.agent import DEFAULT_AGENT_ID, load_agent
from triage.automation.allowlist import load_allowed_suffixes
from triage.automation.catalog import approved_only, load_catalog
from triage.automation.link_check import LinkCheckOptions, check_link
from triage.automation.llm_client import OpenAICompatibleClient
from triage.automation.resolver import CatalogLookup, build_default_lookup
from triage.automation.response import build_response
from triage.classification import classify, triage_decision
from triage.classification.decision import ACTION_HUMAN_REVIEW
from triage.classification.model import ModelNotAvailableError, load, train
from triage.classification.preprocessing import normalize
from triage.config import load_settings
from triage.evaluation import (
    accuracy,
    automatic_accuracy,
    automatic_coverage,
    confusion_matrix,
    per_class_metrics,
    review_rate,
)
from triage.ingestion import Request
from triage.logging_setup import configure_logging, shutdown_logging
from triage.pipeline import Deps, process_request, run_batch
from triage.pipeline.report import render_markdown
from triage.storage import Repository

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

WORK_DIR = Path("output") / "notebook-evidence"  # temporário e ignorado pelo Git; removido na última célula
shutil.rmtree(WORK_DIR, ignore_errors=True)
WORK_DIR.mkdir(parents=True)

# Configuração base: lê o ambiente, mas sem `.env` e com o LLM sempre desligado.
BASE_SETTINGS = dataclasses.replace(
    load_settings(Path("sem.env")), llm_enabled=False, output_dir=str(WORK_DIR), log_level="INFO"
)
LABELED_CSV = Path("data/labeled/requests_labeled.csv")
print("Pasta de trabalho temporária:", WORK_DIR)

Pasta de trabalho temporária: output\notebook-evidence


## 1. Execução ponta a ponta

O robô lê todos os arquivos de `data/samples` (txt, csv e json, incluindo arquivos propositalmente defeituosos),
anonimiza, classifica, resolve o link do catálogo (verificado de verdade na rede) e grava o relatório.
O modelo é treinado aqui, em memória, só com `split=train` (o mesmo que `python scripts/train_model.py` faz).
Se a rede estiver indisponível, os links aparecem como "fora do ar" e os pedidos vão à revisão: é o caminho seguro.

In [2]:
model = train(LABELED_CSV)
e2e_settings = dataclasses.replace(BASE_SETTINGS, output_dir=str(WORK_DIR / "e2e"))
configure_logging(e2e_settings)
deps = Deps(
    repository=Repository(WORK_DIR / "e2e" / "triage.db"),
    lookup=build_default_lookup(),
    settings=e2e_settings,
    model=model,
    llm_client=None,
)
report = run_batch("data/samples", deps)
shutdown_logging()
print(render_markdown(report))

2026-11-10 19:20:32,139 WARNING triage.ingestion Pedido rejeitado: encoding_quebrado.txt (unreadable_encoding)


2026-11-10 19:20:32,141 WARNING triage.ingestion Pedido rejeitado: gigante.txt (too_large)


2026-11-10 19:20:32,144 WARNING triage.ingestion Pedido rejeitado: invalido.json (invalid_json)


2026-11-10 19:20:32,146 WARNING triage.ingestion Pedido rejeitado: pedidos_bons.json#3 (text_too_short)


2026-11-10 19:20:32,147 WARNING triage.ingestion Pedido rejeitado: pedidos_bons.json#4 (missing_text)


2026-11-10 19:20:32,148 WARNING triage.ingestion Pedido rejeitado: sem_coluna_text.csv (missing_text_column)


2026-11-10 19:20:32,149 WARNING triage.ingestion Pedido rejeitado: vazio.txt (empty_file)


2026-11-10 19:20:38,900 INFO triage.pipeline Relatório gravado em output\notebook-evidence\e2e\reports\run-20261110T222032-1.json


2026-11-10 19:20:38,900 INFO triage.pipeline Execução 1 concluída: {'answered': 4, 'queued_for_review': 4, 'rejected': 7, 'skipped_duplicate': 0, 'error': 0}


# Relatório da execução 1

- Início: 2026-11-10T22:20:32.128905+00:00
- Fim: 2026-11-10T22:20:38.891133+00:00
- Duração: 6.762 s

## Totais por status

| Item | Quantidade |
|---|---|
| respondidos | 4 |
| encaminhados à revisão | 4 |
| rejeitados na ingestão | 7 |
| já processados (ignorados) | 0 |
| com erro | 0 |

## Por categoria

| Item | Quantidade |
|---|---|
| birth_certificate | 1 |
| driver_license | 1 |
| id_card | 1 |
| other | 3 |
| social_security | 1 |
| voter_id | 1 |

## Por motivo de revisão

| Item | Quantidade |
|---|---|
| invalid_input | 7 |
| low_confidence | 1 |
| out_of_catalog | 3 |

## Links do catálogo (por pedido)

| Item | Quantidade |
|---|---|
| no ar | 4 |
| fora do ar | 0 |

## LLM

- Respostas reformuladas pelo LLM: 0
- Fallbacks para o template: nenhum



Rodar a mesma pasta de novo não refaz nada (idempotência): os pedidos já gravados são ignorados.

In [3]:
configure_logging(e2e_settings)
second = run_batch("data/samples", deps)
shutdown_logging()
print("Segunda execução, totais:", second.totals)

2026-11-10 19:20:38,921 WARNING triage.ingestion Pedido rejeitado: encoding_quebrado.txt (unreadable_encoding)


2026-11-10 19:20:38,924 WARNING triage.ingestion Pedido rejeitado: gigante.txt (too_large)


2026-11-10 19:20:38,925 WARNING triage.ingestion Pedido rejeitado: invalido.json (invalid_json)


2026-11-10 19:20:38,926 WARNING triage.ingestion Pedido rejeitado: pedidos_bons.json#3 (text_too_short)


2026-11-10 19:20:38,926 WARNING triage.ingestion Pedido rejeitado: pedidos_bons.json#4 (missing_text)


2026-11-10 19:20:38,927 WARNING triage.ingestion Pedido rejeitado: sem_coluna_text.csv (missing_text_column)


2026-11-10 19:20:38,929 WARNING triage.ingestion Pedido rejeitado: vazio.txt (empty_file)


2026-11-10 19:20:38,953 INFO triage.pipeline Relatório gravado em output\notebook-evidence\e2e\reports\run-20261110T222038-2.json


2026-11-10 19:20:38,954 INFO triage.pipeline Execução 2 concluída: {'answered': 0, 'queued_for_review': 0, 'rejected': 0, 'skipped_duplicate': 15, 'error': 0}


Segunda execução, totais: {'answered': 0, 'queued_for_review': 0, 'rejected': 0, 'skipped_duplicate': 15, 'error': 0}


## 2. Métricas de classificação (conjunto de teste, sem LLM)

- **Conjunto:** `split=test` de `data/labeled/requests_labeled.csv`, gravado no próprio arquivo (6 pedidos por
  categoria, 9 categorias). O modelo foi treinado **só** com `split=train`, que tem pelo menos 40 pedidos por
  categoria: a célula abaixo confere que nenhum `id` de teste aparece no treino.
- **Sem LLM:** o LLM só reformula a resposta já decidida (ADR 005); classificar e decidir a revisão são
  determinísticos, então o acerto medido não depende dele.
- **Decisão:** `classify` + `triage_decision` com os limiares padrão (`CONFIDENCE_MIN=0.60`, `AMBIGUITY_MARGIN=0.15`).
- **Amostra:** os números descrevem esta amostra sintética, não pedidos reais.

In [4]:
labeled = pd.read_csv(LABELED_CSV, encoding="utf-8")
train_ids = set(labeled.loc[labeled["split"] == "train", "id"])
test = labeled[labeled["split"] == "test"].reset_index(drop=True)
assert not train_ids & set(test["id"]), "vazamento: pedido de teste no treino"


def triage_row(row):
    classification = classify(row.text, model)
    decision = triage_decision(row.text, classification, BASE_SETTINGS)
    return {
        "id": row.id,
        "text": row.text,
        "real": row.label,
        "previsto": classification.label,
        "fonte": classification.source,
        "confianca": round(decision.confidence, 2),
        "revisao": decision.action == ACTION_HUMAN_REVIEW,
        "motivo": decision.reason.value if decision.reason else "",
    }


results = pd.DataFrame([triage_row(row) for row in test.itertuples()])
labels = sorted(model.classes_)
y_true, y_pred, reviewed = results["real"].tolist(), results["previsto"].tolist(), results["revisao"].tolist()

print(f"Tamanho da amostra de teste: {len(results)} pedidos ({len(results) // len(labels)} por categoria)")
print(f"Acurácia geral: {accuracy(y_true, y_pred):.1%}  (um pedido a mais ou a menos muda {1 / len(results):.1%})")
print(f"Taxa de pedidos enviados à revisão humana: {review_rate(reviewed):.1%}")
print(f"Cobertura automática (respondidos sem revisão): {automatic_coverage(reviewed):.1%}")
answered = len(results) - sum(reviewed)
print(f"Acurácia entre os respondidos sozinhos: {automatic_accuracy(y_true, y_pred, reviewed):.1%}  ({answered} pedidos)")

Tamanho da amostra de teste: 54 pedidos (6 por categoria)
Acurácia geral: 88.9%  (um pedido a mais ou a menos muda 1.9%)
Taxa de pedidos enviados à revisão humana: 33.3%
Cobertura automática (respondidos sem revisão): 66.7%
Acurácia entre os respondidos sozinhos: 100.0%  (36 pedidos)


### Antes e depois da ampliação do treino

Mesmo `split=test` (54 pedidos, nenhuma linha mudou) e mesma decisão; só o treino do modelo muda: **antes**, as 126
linhas de treino originais (14 por categoria); **depois**, essas mais as linhas novas (pelo menos 40 por categoria).
A coluna de confiança mostra quanto o modelo "acredita" nos acertos e nos erros. Isto descreve esta amostra
sintética, não pedidos reais.

In [5]:
ORIGINAL_ROWS = 180  # as linhas do conjunto original (126 de treino e 54 de teste), que a ampliação não altera
ORIGINAL_CSV = WORK_DIR / "original.csv"
labeled.head(ORIGINAL_ROWS).to_csv(ORIGINAL_CSV, index=False, encoding="utf-8")
model_before = train(ORIGINAL_CSV)


def summarize(candidate, train_size):
    """Mede o modelo `candidate` no mesmo teste, com a mesma decisão e os mesmos limiares."""
    outcomes = []
    for row in test.itertuples():
        found = classify(row.text, candidate)
        decision = triage_decision(row.text, found, BASE_SETTINGS)
        outcomes.append((row.label, found.label, decision.action == ACTION_HUMAN_REVIEW, decision.confidence))
    real, predicted, sent, confidence = (list(column) for column in zip(*outcomes))
    hit_confidence = [value for r, p, value in zip(real, predicted, confidence) if r == p]
    error_confidence = [value for r, p, value in zip(real, predicted, confidence) if r != p]
    return {
        "pedidos de treino": train_size,
        "acurácia geral": f"{accuracy(real, predicted):.1%}",
        "revisão humana": f"{review_rate(sent):.1%}",
        "cobertura automática": f"{automatic_coverage(sent):.1%}",
        "acurácia entre os respondidos": f"{automatic_accuracy(real, predicted, sent):.1%}",
        "confiança média dos acertos": f"{sum(hit_confidence) / len(hit_confidence):.2f}",
        "confiança média dos erros": f"{sum(error_confidence) / len(error_confidence):.2f}",
    }


train_rows_before = int((labeled.head(ORIGINAL_ROWS)["split"] == "train").sum())
train_rows_after = int((labeled["split"] == "train").sum())
comparison = pd.DataFrame(
    {
        f"antes ({train_rows_before} de treino)": summarize(model_before, train_rows_before),
        f"depois ({train_rows_after} de treino)": summarize(model, train_rows_after),
    }
)
print(f"Mesmo teste nas duas colunas: {len(test)} pedidos, 6 por categoria. Treino por categoria depois:")
print(labeled[labeled["split"] == "train"]["label"].value_counts().sort_index().to_string())
comparison

Mesmo teste nas duas colunas: 54 pedidos, 6 por categoria. Treino por categoria depois:


label
birth_certificate    40
cpf_registration     40
driver_license       40
id_card              40
other                44
property_tax         40
social_benefits      40
social_security      40
voter_id             40


,antes (126 de treino),depois (364 de treino)
pedidos de treino,126,364
acurácia geral,81.5%,88.9%
revisão humana,42.6%,33.3%
cobertura automática,57.4%,66.7%
acurácia entre os respondidos,100.0%,100.0%
confiança média dos acertos,0.73,0.77
confiança média dos erros,0.26,0.31


In [6]:
by_class = per_class_metrics(y_true, y_pred, labels)
table = pd.DataFrame(
    [
        {"classe": label, "precisão": m.precision, "recall": m.recall, "F1": m.f1, "pedidos (suporte)": m.support}
        for label, m in by_class.items()
    ]
).set_index("classe")
table.round(2)

,precisão,recall,F1,pedidos (suporte)
classe,,,,
birth_certificate,1.00,1.00,1.00,6
cpf_registration,0.86,1.00,0.92,6
driver_license,0.71,0.83,0.77,6
id_card,1.00,1.00,1.00,6
other,0.71,0.83,0.77,6
property_tax,1.00,1.00,1.00,6
social_benefits,0.86,1.00,0.92,6
social_security,1.00,0.67,0.80,6
voter_id,1.00,0.67,0.80,6


In [7]:
matrix = pd.DataFrame(confusion_matrix(y_true, y_pred, labels), index=labels, columns=labels)
matrix.index.name = "real ↓ / previsto →"
matrix

,birth_certificate,cpf_registration,driver_license,id_card,other,property_tax,social_benefits,social_security,voter_id
real ↓ / previsto →,,,,,,,,,
birth_certificate,6,0,0,0,0,0,0,0,0
cpf_registration,0,6,0,0,0,0,0,0,0
driver_license,0,1,5,0,0,0,0,0,0
id_card,0,0,0,6,0,0,0,0,0
other,0,0,1,0,5,0,0,0,0
property_tax,0,0,0,0,0,6,0,0,0
social_benefits,0,0,0,0,0,0,6,0,0
social_security,0,0,1,0,1,0,0,4,0
voter_id,0,0,0,0,1,0,1,0,4


In [8]:
print("Motivos de revisão entre os pedidos do teste:")
print(results.loc[results["revisao"], "motivo"].value_counts().to_string())
print()
print("Pedidos enviados à revisão, por categoria real:")
print(results.groupby("real")["revisao"].sum().astype(int).to_string())

Motivos de revisão entre os pedidos do teste:
motivo
low_confidence    10
out_of_catalog     7
urgent             1

Pedidos enviados à revisão, por categoria real:
real
birth_certificate    1
cpf_registration     0
driver_license       2
id_card              1
other                6
property_tax         1
social_benefits      2
social_security      3
voter_id             2


### Limiares: calibração por validação cruzada só no treino

Os limiares **não** são escolhidos olhando o `split=test` (isso mediria o ajuste, não o modelo). A calibração usa
validação cruzada estratificada de 5 partes **só sobre as linhas de treino**: cada pedido recebe a probabilidade
de um modelo que não o viu, e a decisão de revisão é a do robô (regra única, urgência, `other`, confiança e folga).
Regra de mudança: baixar `CONFIDENCE_MIN` só se a cobertura subir sem a acurácia entre os respondidos ficar pior
que a do padrão (0,60); `AMBIGUITY_MARGIN` idem.

In [9]:
from triage.evaluation.calibration import (
    classifications_from_probabilities,
    clone_fit_predict,
    confidence_grid,
    margin_grid,
    mean_confidence_by_outcome,
    out_of_fold_probabilities,
    threshold_table,
)

train_rows = labeled[labeled["split"] == "train"].reset_index(drop=True)
cv_texts, cv_labels = train_rows["text"].tolist(), train_rows["label"].tolist()
cv_probabilities = out_of_fold_probabilities(cv_texts, cv_labels, clone_fit_predict(model))
cv_found = classifications_from_probabilities(cv_texts, cv_probabilities)
cv_predicted = [found.label for found in cv_found]
print(f"Validação cruzada (5 partes) só no treino: {len(cv_texts)} pedidos")
print(f"Acurácia geral fora da amostra: {accuracy(cv_labels, cv_predicted):.1%}")
hit_mean, error_mean = mean_confidence_by_outcome(cv_texts, cv_labels, cv_found, BASE_SETTINGS)
print(f"Confiança média dos acertos: {hit_mean:.2f} | dos erros: {error_mean:.2f}")


def limit_table(rows):
    table = pd.DataFrame([dataclasses.asdict(row) for row in rows])
    table["coverage"] = table["coverage"].map("{:.1%}".format)
    table["accuracy"] = table["accuracy"].map("{:.1%}".format)
    return table.rename(
        columns={
            "confidence_min": "CONFIDENCE_MIN",
            "ambiguity_margin": "AMBIGUITY_MARGIN",
            "coverage": "cobertura automática",
            "accuracy": "acurácia entre os respondidos",
        }
    ).set_index("CONFIDENCE_MIN")


confidence_grid_values = [round(0.40 + 0.05 * step, 2) for step in range(9)]
confidence_rows = threshold_table(cv_texts, cv_labels, cv_found, confidence_grid(BASE_SETTINGS, confidence_grid_values))
limit_table(confidence_rows)

Validação cruzada (5 partes) só no treino: 364 pedidos
Acurácia geral fora da amostra: 92.9%
Confiança média dos acertos: 0.77 | dos erros: 0.41


,AMBIGUITY_MARGIN,cobertura automática,acurácia entre os respondidos
CONFIDENCE_MIN,,,
0.40,0.15,76.9%,96.8%
0.45,0.15,75.3%,96.7%
0.50,0.15,72.0%,97.3%
0.55,0.15,71.2%,97.7%
0.60,0.15,69.2%,98.0%
0.65,0.15,67.9%,98.0%
0.70,0.15,65.9%,98.3%
0.75,0.15,64.8%,98.7%
0.80,0.15,63.7%,98.7%


In [10]:
margin_rows = threshold_table(cv_texts, cv_labels, cv_found, margin_grid(BASE_SETTINGS, [0.05, 0.10, 0.15, 0.20, 0.25]))
limit_table(margin_rows).reset_index().set_index("AMBIGUITY_MARGIN")

,CONFIDENCE_MIN,cobertura automática,acurácia entre os respondidos
AMBIGUITY_MARGIN,,,
0.05,0.6,69.2%,98.0%
0.10,0.6,69.2%,98.0%
0.15,0.6,69.2%,98.0%
0.20,0.6,69.2%,98.0%
0.25,0.6,69.2%,98.0%


**Decisão: os limiares se mantiveram** (`CONFIDENCE_MIN=0.60`, `AMBIGUITY_MARGIN=0.15`).

- `CONFIDENCE_MIN`: baixar o mínimo aumenta a cobertura automática (69,2% em 0,60 para 71,2% em 0,55 e 76,9% em
  0,40), mas a acurácia entre os respondidos cai (98,0% em 0,60 para 97,7% em 0,55 e 96,8% em 0,40). A regra
  exigia cobertura maior **sem** acurácia pior que a do padrão; nenhum valor abaixo de 0,60 cumpre, então 0,60 fica.
  Valores acima de 0,60 reduzem a cobertura e não foram adotados.
- `AMBIGUITY_MARGIN`: as cinco linhas da tabela são idênticas. Com confiança mínima de 0,60, a folga para a segunda
  categoria já é de pelo menos 0,20 (as probabilidades somam 1), então a folga de 0,15 não chega a barrar nada
  nesta amostra; não há ganho a medir e o valor se mantém.
- A validação cruzada usa só o treino; o `split=test` não entrou em nenhuma escolha. Os números descrevem esta
  amostra sintética, não pedidos reais.

## 3. Análise de erros

Todos os pedidos em que a categoria prevista difere da real. A coluna **fora do vocabulário** lista os termos do
pedido que o TF-IDF nunca viu no treino (medido, não suposto); **destino** diz o que o robô fez com o pedido.

In [11]:
vocabulary = set(model.named_steps["tfidf"].vocabulary_)


def unseen_terms(text):
    tokens = normalize(text).split()
    return ", ".join(sorted({token for token in tokens if token not in vocabulary}))


errors = results[results["real"] != results["previsto"]].copy()
errors["fora do vocabulário"] = errors["text"].map(unseen_terms)
errors["destino"] = errors["motivo"].map(lambda reason: f"revisão humana ({reason})" if reason else "respondido sozinho")
errors[["id", "text", "real", "previsto", "fonte", "confianca", "fora do vocabulário", "destino"]].set_index("id")

,text,real,previsto,fonte,confianca,fora do vocabulário,destino
id,,,,,,,
L010,Preciso do CPF [CPF] ou de outro documento pra abrir o processo da carteira de motorista?,driver_license,cpf_registration,model,0.39,,revisão humana (low_confidence)
L094,"não votei nas últimas eleições, tem multa? como pagar",voter_id,other,model,0.32,"eleicoes, ultimas, votei",revisão humana (out_of_catalog)
L100,"mesário voluntário, como me inscrever",voter_id,social_benefits,model,0.21,"mesario, voluntario",revisão humana (low_confidence)
L130,como emitir o meu CNIS,social_security,other,model,0.35,,revisão humana (out_of_catalog)
L140,como fazer a prova de vida do aposentado,social_security,driver_license,model,0.39,vida,revisão humana (low_confidence)
L164,"preciso renovar meu passaporte, perdi o prazo",other,driver_license,model,0.20,,revisão humana (low_confidence)


In [12]:
print(f"Erros de classificação: {len(errors)} de {len(results)} pedidos ({len(errors) / len(results):.1%})")
print(f"Erros que o robô respondeu sozinho (sem revisão): {int((~errors['revisao']).sum())}")
print(f"Erros encaminhados à revisão humana: {int(errors['revisao'].sum())}")

Erros de classificação: 6 de 54 pedidos (11.1%)
Erros que o robô respondeu sozinho (sem revisão): 0
Erros encaminhados à revisão humana: 6


**Agrupamento e causa provável** (interpretação a partir da tabela acima; a causa é uma hipótese, não foi testada
com experimento próprio):

| Grupo | Pedidos | Causa provável |
|---|---|---|
| Assunto que o treino cobre pouco | L094, L100 (título de eleitor); L130 (CNIS); L140 (aposentadoria) | O pedido usa termos específicos (não votar e a multa, mesário, CNIS, prova de vida), raros ou ausentes no treino; com poucos termos conhecidos, a probabilidade se espalha e a confiança fica baixa (0,21 a 0,39). |
| Duas categorias no mesmo pedido | L010 (CPF para abrir o processo da carteira de motorista) | O texto cita CPF e carteira de motorista; o modelo escolheu a outra categoria com confiança 0,39. É a ambiguidade que o conjunto traz de propósito. |
| Categoria `other` heterogênea | L164 (passaporte) | `other` mistura assuntos sem relação entre si; palavras soltas ("renovar", "prazo") aproximam o pedido de um serviço do catálogo. |

Em todos os casos a confiança ficou abaixo do limite ou a categoria prevista foi `other`, e por isso o pedido foi
para a revisão humana em vez de receber um link errado.

## 4. Casos de falha tratados

Cada caso provoca a falha de verdade (ou a simula, quando indicado), mostra o **log** gerado (saída de erro da
célula) e o **caminho seguro** que o robô seguiu. A execução nunca é interrompida e nenhum log carrega o texto do
cidadão.

In [13]:
def case_settings(name):
    folder = WORK_DIR / name
    settings = dataclasses.replace(BASE_SETTINGS, output_dir=str(folder))
    configure_logging(settings)
    return settings, folder


def fresh_deps(settings, folder, **overrides):
    values = {
        "repository": Repository(folder / "triage.db"),
        "lookup": build_default_lookup(),
        "settings": settings,
        "model": model,
        "llm_client": None,
    }
    return Deps(**{**values, **overrides})


CLEAR_REQUEST = Request("DEMO-1", "Como faço para renovar a minha CNH?", "notebook")

### 4.1 Arquivo ausente (pasta de entrada inexistente)

In [14]:
settings, folder = case_settings("ausente")
exit_code = run.main(["--input", "pasta_que_nao_existe", "--output", str(folder), "--no-llm"])
print("Código de saída:", exit_code, "(2 = entrada inválida; a execução terminou sem exceção)")

2026-11-10 19:20:41,169 ERROR triage.run Pasta de entrada não encontrada: pasta_que_nao_existe


Código de saída: 2 (2 = entrada inválida; a execução terminou sem exceção)


### 4.2 JSON inválido

`data/samples/invalido.json` tem o JSON truncado. O arquivo é rejeitado, registrado e enviado à fila de revisão com
o motivo `invalid_input`; os demais arquivos da pasta seguem normalmente.

In [15]:
settings, folder = case_settings("json")
input_dir = folder / "entrada"
input_dir.mkdir()
shutil.copy("data/samples/invalido.json", input_dir)
shutil.copy("data/samples/pedido_bom.txt", input_dir)
json_report = run_batch(input_dir, fresh_deps(settings, folder))
shutdown_logging()
print("Totais:", json_report.totals)
print("Motivos de revisão:", json_report.by_review_reason)

2026-11-10 19:20:41,225 WARNING triage.ingestion Pedido rejeitado: invalido.json (invalid_json)


2026-11-10 19:20:42,753 INFO triage.pipeline Relatório gravado em output\notebook-evidence\json\reports\run-20261110T222041-1.json


2026-11-10 19:20:42,755 INFO triage.pipeline Execução 1 concluída: {'answered': 1, 'queued_for_review': 0, 'rejected': 1, 'skipped_duplicate': 0, 'error': 0}


Totais: {'answered': 1, 'queued_for_review': 0, 'rejected': 1, 'skipped_duplicate': 0, 'error': 0}
Motivos de revisão: {'invalid_input': 1}


### 4.3 Link fora do ar (simulado com uma porta local fechada)

O verificador de link real (`check_link`, com timeout e retry com espera) aponta para uma porta local sem serviço.
Ele tenta de novo, desiste sem levantar exceção e o pedido, que seria respondido, vai à revisão humana
(`link_unavailable`): o robô não entrega link que não conseguiu confirmar.

In [16]:
def link_always_down(_url):
    return check_link("http://127.0.0.1:9/", LinkCheckOptions(timeout=0.5, retries=2, backoff=0.1))


settings, folder = case_settings("link")
down_lookup = CatalogLookup(load_catalog(), load_allowed_suffixes(), link_always_down)
result = process_request(CLEAR_REQUEST, fresh_deps(settings, folder, lookup=down_lookup))
shutdown_logging()
print("Desfecho:", result.status, "| categoria:", result.category, "| motivo:", result.reason, "| link:", result.link_status)

2026-11-10 19:20:43,303 WARNING triage.retry Tentativa 1/2 falhou (HTTPConnectionPool(host='127.0.0.1', port=9): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<HTTPConnection(host='127.0.0.1', port=9) at 0x15f4900bb50>, 'Connection to 127.0.0.1 timed out. (connect timeout=0.5)'))); nova tentativa em 0.10s


2026-11-10 19:20:43,913 ERROR triage.retry Tentativa 2/2 falhou; desistindo: HTTPConnectionPool(host='127.0.0.1', port=9): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<HTTPConnection(host='127.0.0.1', port=9) at 0x15f49019050>, 'Connection to 127.0.0.1 timed out. (connect timeout=0.5)'))


2026-11-10 19:20:43,914 WARNING triage.automation Link inacessível (ConnectTimeout)


2026-11-10 19:20:43,915 WARNING triage.automation Link de driver_license indisponível (ConnectTimeout)


Desfecho: queued_for_review | categoria: driver_license | motivo: link_unavailable | link: unavailable


### 4.4 Resposta vazia do LLM (simulada)

A chamada HTTP é substituída por uma resposta 200 com `content` vazio, que o cliente trata como resposta inválida.
A resposta final cai no template do catálogo e o motivo fica registrado. Nenhuma chamada real é feita.

In [17]:
empty_reply = MagicMock(status_code=200)
empty_reply.json.return_value = {"choices": [{"message": {"content": ""}}]}
settings, folder = case_settings("llm")
service = next(entry for entry in approved_only(load_catalog()) if entry.category == "driver_license")
client = OpenAICompatibleClient("http://llm.invalid/v1", "valor-ficticio", ("modelo-de-teste",), load_agent(DEFAULT_AGENT_ID))
with patch("triage.automation.llm_client.requests.post", return_value=empty_reply):
    response = build_response(service, client)
shutdown_logging()
print("Usou o LLM:", response.used_llm, "| motivo do fallback:", response.fallback_reason)
print("Resposta entregue (template):", response.text)

2026-11-10 19:20:43,945 WARNING triage.automation Resposta inválida do LLM (conteúdo vazio ou não textual); usando o template


Usou o LLM: False | motivo do fallback: llm_invalid_response
Resposta entregue (template): Para resolver o seu pedido, o serviço indicado é: Carteira Nacional de Habilitação (CNH).
Órgão responsável: Secretaria Nacional de Trânsito (Senatran) e Detrans estaduais.
Abrangência: Brasil (o atendimento é feito pelo Detran do seu estado).
A página do gov.br explica como emitir a CNH digital. Primeira habilitação, renovação e segunda via são atendidas pelo Detran do seu estado, com login gov.br.
Acesse o link oficial:
Emitir a CNH digital: https://www.gov.br/pt-br/servicos/emitir-a-carteira-nacional-de-habilitacao-digital-cnh-e


### 4.5 Modelo ausente

O arquivo do modelo não existe: `load` levanta `ModelNotAvailableError`, o robô segue **sem modelo** e todo pedido
vai à revisão humana, em vez de a execução parar.

In [18]:
settings, folder = case_settings("modelo")
try:
    load(folder / "nao_existe.joblib")
except ModelNotAvailableError as error:
    logging.getLogger("triage.pipeline").warning(
        "Modelo indisponível (%s); todos os pedidos irão à revisão humana", type(error).__name__
    )
result = process_request(CLEAR_REQUEST, fresh_deps(settings, folder, model=None))
shutdown_logging()
print("Desfecho:", result.status, "| motivo:", result.reason)

2026-11-10 19:20:43,962 WARNING triage.pipeline Modelo indisponível (ModelNotAvailableError); todos os pedidos irão à revisão humana


Desfecho: queued_for_review | motivo: low_confidence


## 5. Localizador de fontes aprovadas (sem LLM)

Pedidos como "alvará" ou "buraco na rua" caem em `other` no classificador. O `ServiceFinder` procura, entre as entradas
**aprovadas** do catálogo, a que melhor atende o texto anonimizado do pedido: TF-IDF e cosseno sobre `keywords`,
`display_name` e `agency`, sem LLM e sem rede (ADR 005). Ele só devolve a **entrada**; nunca produz link nem telefone.
Devolve nada (`None`) quando a melhor pontuação fica abaixo de `FINDER_MIN_SCORE` ou a folga para a segunda melhor
fica abaixo de `FINDER_MARGIN`.

- **Conjunto:** `data/labeled/finder_requests.csv`, **100% sintético**, 96 pedidos escritos à mão: 60 com fonte
  esperada (32 entradas diferentes, inclusive 10 que citam um serviço de outra esfera) e 36 `none` (assunto que
  nenhuma fonte aprovada atende, com 12 negativos que dividem vocabulário com o catálogo e 2 de outra cidade).
- **Divisão:** `split=train` (48) para ajustar os dois limiares e `split=test` (48), só para reportar.
- **Métricas:** cobertura (devolveu fonte), acerto entre os que devolveram fonte e falso positivo (devolveu fonte
  para um pedido `none`). O padrão privilegia falso positivo baixo: é melhor dizer "não sei" do que apontar a fonte errada.
- **Amostra:** os números descrevem esta amostra sintética, não pedidos reais.

In [20]:
from triage.automation.finder import ServiceFinder
from triage.evaluation.finder_metrics import NO_SOURCE, score_finder, threshold_table

finder_rows = pd.read_csv("data/labeled/finder_requests.csv", encoding="utf-8")
finder_train = finder_rows[finder_rows["split"] == "train"].reset_index(drop=True)
finder_test = finder_rows[finder_rows["split"] == "test"].reset_index(drop=True)
assert not set(finder_train["id"]) & set(finder_test["id"]), "vazamento: pedido de teste no treino"
approved = approved_only(load_catalog())
assert set(finder_rows["expected_id"]) - {NO_SOURCE} <= {entry.id for entry in approved}, "fonte esperada não aprovada"

# `best` devolve a melhor entrada sem aplicar limiar; assim a tabela varre os limiares sem refazer a busca.
open_settings = dataclasses.replace(BASE_SETTINGS, finder_min_score=0.0, finder_margin=0.0)
finder = ServiceFinder(approved, open_settings)
print(f"Catálogo: {len(approved)} entradas aprovadas | treino: {len(finder_train)} pedidos | teste: {len(finder_test)} pedidos")


def sweep_table(rows, min_scores, margins):
    matches = [finder.best(text) for text in rows["text"]]
    table = pd.DataFrame(
        [
            {
                "FINDER_MIN_SCORE": row.min_score,
                "FINDER_MARGIN": row.margin,
                "cobertura": f"{row.scores.coverage:.1%}",
                "acerto entre os respondidos": f"{row.scores.answered_accuracy:.1%}"
                if row.scores.answered_accuracy is not None
                else "-",
                "falso positivo": f"{row.scores.false_positive_rate:.1%}",
            }
            for row in threshold_table(rows["expected_id"].tolist(), matches, min_scores, margins)
        ]
    )
    return table.set_index(["FINDER_MIN_SCORE", "FINDER_MARGIN"])


MIN_SCORES = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
MARGINS = [0.0, 0.10, 0.15, 0.20]
print("Varredura dos limiares só no TREINO (não olha o teste):")
sweep_table(finder_train, MIN_SCORES, MARGINS)

Catálogo: 51 entradas aprovadas | treino: 48 pedidos | teste: 48 pedidos
Varredura dos limiares só no TREINO (não olha o teste):


cobertura acerto entre os respondidos falso positivo
FINDER_MIN_SCORE FINDER_MARGIN                                                     
0.15             0.00              75.0%                       75.0%          33.3%
                 0.10              62.5%                       86.7%          16.7%
                 0.15              54.2%                       88.5%          11.1%
                 0.20              43.8%                       90.5%           5.6%
0.20             0.00              62.5%                       83.3%          11.1%
                 0.10              56.2%                       88.9%          11.1%
                 0.15              47.9%                       91.3%           5.6%
                 0.20              43.8%                       90.5%           5.6%
0.25             0.00              58.3%                       82.1%          11.1%
                 0.10              52.1%                       88.0%          11.1%
                 0.15              45.8%                       90.9%           5.6%
                 0.20              43.8%                       90.5%           5.6%
0.30             0.00              54.2%                       80.8%          11.1%
                 0.10              47.9%                       87.0%          11.1%
                 0.15              43.8%                       90.5%           5.6%
                 0.20              41.7%                       90.0%           5.6%
0.35             0.00              37.5%                       88.9%           5.6%
                 0.10              35.4%                       88.2%           5.6%
                 0.15              33.3%                       93.8%           0.0%
                 0.20              31.2%                       93.3%           0.0%
0.40             0.00              27.1%                       84.6%           5.6%
                 0.10              27.1%                       84.6%           5.6%
                 0.15              25.0%                       91.7%           0.0%
                 0.20              25.0%                       91.7%           0.0%

**Escolha dos padrões (feita só no treino).** Entre os pares com no máximo 1 falso positivo em 18 pedidos `none` e acerto
de pelo menos 90% entre os respondidos, ficou o de maior cobertura cujo vizinho (±0,05 na pontuação) também cumpre
a regra: `FINDER_MIN_SCORE=0.25` e `FINDER_MARGIN=0.15`. Zerar o falso positivo no treino exigiria
`FINDER_MIN_SCORE` de 0,35 ou mais, só para barrar uma linha (o pedido sobre buraco em Curitiba) e com a cobertura
caindo para 33%; esse ajuste a uma linha não foi feito.

In [21]:
def report(rows, name):
    matches = [finder.best(text) for text in rows["text"]]
    chosen = ServiceFinder(approved, BASE_SETTINGS)
    found = [chosen.find(text) for text in rows["text"]]
    predicted = [match.entry.id if match else None for match in found]
    scores = score_finder(rows["expected_id"].tolist(), predicted)
    answered = sum(item is not None for item in predicted)
    hits = sum(real == got for real, got in zip(rows["expected_id"], predicted, strict=True) if got is not None)
    unanswerable = int((rows["expected_id"] == NO_SOURCE).sum())
    print(f"{name}: {scores.size} pedidos | cobertura {scores.coverage:.1%} ({answered} com fonte)")
    print(f"  acerto entre os respondidos: {scores.answered_accuracy:.1%} ({hits} de {answered})")
    wrong = round(scores.false_positive_rate * unanswerable)
    print(f"  falso positivo: {scores.false_positive_rate:.1%} ({wrong} de {unanswerable} pedidos `none`)")
    rows = rows.assign(devolvido=[item or "-" for item in predicted])
    rows = rows.assign(
        pontuacao=[round(match.score, 2) if match else 0.0 for match in matches],
        folga=[round(match.margin, 2) if match else 0.0 for match in matches],
    )
    return rows[(rows["devolvido"] != "-") & (rows["devolvido"] != rows["expected_id"])]


print(f"Padrões: FINDER_MIN_SCORE={BASE_SETTINGS.finder_min_score} | FINDER_MARGIN={BASE_SETTINGS.finder_margin}")
train_errors = report(finder_train, "TREINO")
test_errors = report(finder_test, "TESTE")
print("\nFonte errada no TESTE (devolveu uma fonte que não é a esperada):")
test_errors[["id", "text", "expected_id", "devolvido", "pontuacao", "folga"]].set_index("id")

Padrões: FINDER_MIN_SCORE=0.25 | FINDER_MARGIN=0.15
TREINO: 48 pedidos | cobertura 45.8% (22 com fonte)
  acerto entre os respondidos: 90.9% (20 de 22)
  falso positivo: 5.6% (1 de 18 pedidos `none`)


TESTE: 48 pedidos | cobertura 47.9% (23 com fonte)
  acerto entre os respondidos: 87.0% (20 de 23)
  falso positivo: 16.7% (3 de 18 pedidos `none`)

Fonte errada no TESTE (devolveu uma fonte que não é a esperada):


,text,expected_id,devolvido,pontuacao,folga
id,,,,,
F080,tenho uma dúvida sobre a mensalidade da minha faculdade particular,none,higher_education_access,0.35,0.35
F090,como cuidar da saúde mental no trabalho,none,labor_ministry,0.34,0.15
F096,como pago o ipva do meu carro no rio de janeiro,none,sefaz_sp_ipva,0.38,0.38


**Leitura dos números** (amostra sintética de 48 pedidos por divisão; um pedido vale cerca de 2 pontos percentuais):

| Divisão | Cobertura | Acerto entre os respondidos | Falso positivo |
|---|---|---|---|
| Treino (48) | 45,8% (22 com fonte) | 90,9% (20 de 22) | 5,6% (1 de 18 `none`) |
| Teste (48) | 47,9% (23 com fonte) | 87,0% (20 de 23) | 16,7% (3 de 18 `none`) |

- **O custo de privilegiar o "não sei":** o localizador deixa sem fonte pouco mais da metade dos pedidos (no teste,
  10 dos 30 pedidos que têm fonte esperada ficam sem resposta). Nenhum pedido com fonte esperada recebeu uma fonte
  errada no teste; no treino, 1 (um pedido com dois assuntos, vacina e creche).
- **O falso positivo não ficou baixo no teste:** 3 de 18 pedidos `none` receberam uma fonte, contra 1 de 18 no treino.
  Dois deles dividem palavras com o catálogo (faculdade particular, saúde mental no trabalho) e um é de outra
  cidade (IPVA do Rio de Janeiro): o piloto cobre só São Paulo e o localizador não distingue a cidade. O
  teste não foi usado para mudar os limiares; subir `FINDER_MIN_SCORE` para 0,35 reduziria o falso positivo
  no teste, mas essa escolha pertence a uma nova rodada com amostra nova, não a esta.
- **Limites:** a pontuação mede palavras em comum, não o sentido; pedido com dois assuntos de esferas diferentes
  e pedido de outra cidade ou estado são os casos frágeis. Os números descrevem esta amostra sintética, não pedidos reais.

## 6. Limites da solução

Escrito só com o que foi medido ou é fato do desenho:

- **Amostra pequena e sintética.** O teste tem 54 pedidos (6 por categoria) e o treino, 364 (pelo menos 40 por
  categoria), todos escritos à mão para este protótipo; a validação cruzada do treino também é sintética. Cada
  pedido do teste vale cerca de 1,9 ponto percentual na acurácia, e 6 pedidos por classe não sustentam conclusão
  estatística por categoria. Os números descrevem esta amostra sintética, não pedidos reais.
- **Sem dados reais.** A anonimização cobre CPF, telefone, e-mail, nome, endereço, CEP, RG, data completa e
  documentos, por regex, contexto e lista de prenomes, e na dúvida mascara. Risco residual: nome fora de contexto e
  fora da lista pode passar; endereço sem logradouro passa; cidade, estado e bairro não são mascarados.
- **Catálogo estático.** São 8 serviços curados à mão; o robô não busca informação ao vivo e o link só vale até o
  órgão mudar a página (por isso o link é verificado a cada execução).
- **Revisão humana alta.** Uma parte grande dos pedidos do teste vai à revisão (seção 2): o robô prefere pedir
  ajuda humana a arriscar um link errado, e isso custa cobertura automática.
- **Ambiguidade.** Pedido com dois assuntos (como CPF e CNH) cai na dúvida e vai à revisão; o robô não pergunta de volta.
- **Sem multi-turno.** Cada pedido é independente; não há conversa nem pedido de esclarecimento.
- **LLM opcional e não medido.** Só reformula a resposta; ficou desligado nas medições e a chamada real a um provedor
  não foi exercitada aqui, só os caminhos de falha simulados.
- **Sem autenticação.** Quem tem acesso à máquina vê a fila de revisão (ADR 003).

In [19]:
shutil.rmtree(WORK_DIR, ignore_errors=True)
if Path("output").is_dir() and not any(Path("output").iterdir()):
    Path("output").rmdir()
print("Pasta temporária removida.")

Pasta temporária removida.
